# Data Loading

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('set_4_st.csv')
df.shape

(25000, 2)

In [5]:
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    25000 non-null  object
 1   label   25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB


,text,label
count,25000,25000
unique,200,2
top,The app experience was wonderful and satisfying.,positive
freq,156,12507


In [7]:
df['clean_text'] = df['text'].str.lower()
df['clean_text'] = df['text'].str.replace(r'\[.*?\]', '', regex=True)
df['clean_text'] = df['clean_text'].str.replace(r'[^\w\s]', '', regex=True)
df

,text,label,clean_text
0,I hated the quality of this watch.,negative,I hated the quality of this watch
1,Terrible and disappointing watch experience.,negative,Terrible and disappointing watch experience
2,I would definitely buy this app again.,positive,I would definitely buy this app again
3,The headphone experience was wonderful and sat...,positive,The headphone experience was wonderful and sat...
4,This course was a huge mistake to buy.,negative,This course was a huge mistake to buy
...,...,...,...
24995,I regret spending money on this laptop.,negative,I regret spending money on this laptop
24996,Worst app ever. Not recommended.,negative,Worst app ever Not recommended
24997,Worst camera ever. Not recommended.,negative,Worst camera ever Not recommended
24998,Brilliant meal design and smooth performance.,positive,Brilliant meal design and smooth performance


# Preprocessing

In [10]:
import spacy
nlp=spacy.load('en_core_web_sm')

In [11]:
def lower_replace (series):
    output = series.str.lower()
    output = output.str.replace(r'\[.*?\]', '', regex=True)
    output = output.str.replace(r'\[^\w\s]', '', regex=True)
    return output

In [12]:
def token_lemma_nonstop(text):
    doc = nlp(text)
    output = [token.lemma_ for token in doc if not token.is_stop]
    output = ' '.join(output)
    return output

In [13]:
lower_replace(df.clean_text).apply(token_lemma_nonstop)

0                              hate quality watch
1         terrible disappointing watch experience
2                              definitely buy app
3          headphone experience wonderful satisfy
4                         course huge mistake buy
                           ...                   
24995                   regret spend money laptop
24996                           bad app recommend
24997                        bad camera recommend
24998    brilliant meal design smooth performance
24999                        hate quality service
Name: clean_text, Length: 25000, dtype: object

# Vectorization and Training

In [18]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

In [20]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_vectorized, y_train)
y_pred = model.predict(X_test_vectorized)

In [22]:
from sklearn.naive_bayes import MultinomialNB
nb_model = MultinomialNB()
nb_model.fit(X_train_vectorized, y_train)
y_nb_pred = nb_model.predict(X_test_vectorized)

# Evaluation

In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print('Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 1.0
Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      2536
    positive       1.00      1.00      1.00      2464

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000

Confusion Matrix:
[[2536    0]
 [   0 2464]]


In [23]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy = accuracy_score(y_test, y_nb_pred)
print(f'Accuracy: {accuracy}')
print('Classification Report:')
print(classification_report(y_test, y_nb_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_nb_pred))

Accuracy: 1.0
Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      2536
    positive       1.00      1.00      1.00      2464

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000

Confusion Matrix:
[[2536    0]
 [   0 2464]]


# Error Analysis - No Errors